# Model 1 — Timeline Drift Predictor (SYNTHETIC)

Component 1 · Anchor AI (J26-IT-366). All results here are **SYNTHETIC**. This notebook trains Model 1 (how long a
task will really take this student), compares it with the simple baselines B0–B3, shows the training as it happens,
and saves the trained model (format 2: median point + earned-trust gate) for the API.

**Cell 1 — setup (run first).** In Google Colab this clones the C1 branch `dev/ravindu` of the team repo into
`/content/AnchorAI` **only if it is not there yet** (running it again never makes a nested copy), goes to that folder,
and installs only what these notebooks need (`components/c1_disruption/notebooks/requirements.txt`: numpy, scipy,
scikit-learn, quantile-forest, joblib, pandas, matplotlib — not the web server). It prints the versions. If a package
that was already loaded had to be replaced (usually numpy), it stops with a clear message: then use
**Runtime → Restart session** and run again from **cell 2** (the next code cell). Outside Colab it uses the repository
the notebook is in and installs nothing. C1 supports Python 3.11–3.13.

In [ ]:
import importlib.metadata as md, os, subprocess, sys

REPO = "https://github.com/DulanjaR/AnchorAI.git"
IN_COLAB = "google.colab" in sys.modules
PACKAGES = {"numpy": "numpy", "scipy": "scipy", "scikit-learn": "sklearn", "quantile-forest": "quantile_forest",
            "joblib": "joblib", "pandas": "pandas", "matplotlib": "matplotlib"}


def repo_root():
    if IN_COLAB:
        return "/content/AnchorAI"
    here = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(here, "components", "c1_disruption")):
        if os.path.dirname(here) == here:
            raise FileNotFoundError("Open this notebook inside the AnchorAI repository (or in Colab).")
        here = os.path.dirname(here)
    return here


def installed():
    versions = {}
    for package in PACKAGES:
        try:
            versions[package] = md.version(package)
        except md.PackageNotFoundError:
            versions[package] = None
    return versions


ROOT = repo_root()
if IN_COLAB and not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "dev/ravindu", REPO, ROOT], check=True)
elif IN_COLAB:
    print("Already cloned at", ROOT, "- not cloning again.")
os.chdir(ROOT)
before = installed()
loaded = {p: getattr(sys.modules[m], "__version__", None) for p, m in PACKAGES.items() if m in sys.modules}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "components/c1_disruption/notebooks/requirements.txt"], check=True)
after = installed()
print("Python", sys.version.split()[0], "| folder", ROOT)
for package, version in after.items():
    print(f"  {package:16} {version}" + ("" if before[package] == version else f"   (was {before[package]})"))
replaced = [p for p, v in loaded.items() if v is not None and v != after[p]]
if replaced:
    print("\n" + "!" * 72 + f"\n{', '.join(replaced)} changed while already loaded.\n"
          "Runtime → Restart session, then run from cell 2 (the next code cell).\n" + "!" * 72)
    raise SystemExit("Restart needed: Runtime → Restart session, then run from cell 2.")

**Cell 2 — start here after a restart.** Goes to the repository folder, makes C1 importable, prints the versions in
use and starts the progress clock (every long step prints a line like `[  12.4s] training fold 3/5…`).

In [ ]:
import os, sys, tempfile

IN_COLAB = "google.colab" in sys.modules
ROOT = "/content/AnchorAI"
if not IN_COLAB:
    ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(ROOT, "components", "c1_disruption")):
        ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Files this notebook writes go here, never into the repository.
OUT = "/content/c1_outputs" if IN_COLAB else os.path.join(tempfile.gettempdir(), "c1_outputs")
os.makedirs(OUT, exist_ok=True)

import joblib, matplotlib, numpy, pandas, quantile_forest, scipy, sklearn
print("Python", sys.version.split()[0], "| numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| quantile-forest", quantile_forest.__version__,
      "| joblib", joblib.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Outputs go to", OUT)
from components.c1_disruption.ml.progress import Progress
progress = Progress()

**Which data.** The SYNTHETIC persona files in `components/c1_disruption/sim/data/`: personas, their special habits,
30 days of planned tasks, each day's schedule and each persona's disruption profile (format:
`components/c1_disruption/sim/README.md`). Everything produced from them is **SYNTHETIC**: it checks that the pipeline
works; it is not a research finding.

In [ ]:
DATA = "components/c1_disruption/sim/data/"
PERSONAS, HABITS = DATA + "SYNTHETIC_personas.csv", DATA + "SYNTHETIC_habits.csv"
TASKS, SCHEDULE = DATA + "SYNTHETIC_tasks_planned.csv", DATA + "SYNTHETIC_schedule.csv"
PROFILE = DATA + "SYNTHETIC_disruption_profile.csv"
from components.c1_disruption.sim.personas import load_personas, load_schedule, load_tasks
from components.c1_disruption.sim.profile import load_profile
progress("loading the SYNTHETIC persona files…")
personas = load_personas(PERSONAS, HABITS)
tasks = load_tasks(TASKS, personas)
schedule = load_schedule(SCHEDULE, personas)
profile = load_profile(PROFILE, personas)
print(len(personas), "personas,", len(tasks), "planned tasks,", len(schedule), "scheduled days")

**Layers B and C.** Each persona gets actual durations (their planted drift × noise) and disruptions from their
profile, from a fixed seed. Records that overlap a disruption are cleaned by the exclusion rule before training.

In [ ]:
from components.c1_disruption.sim.generate import generate
from components.c1_disruption.ml.drift.data import clean
progress("generating actual durations (planted drift × noise) and disruptions, seed 11…")
world = generate(personas, tasks, seed=11, variant="planted", schedule=schedule, profile=profile)
records = clean(world.records)
print(len(world.records), "records,", len(records), "after the exclusion rule; dataset =", world.dataset)

**One training run, step by step.** This is what "training Model 1" means: the forest is trained 5 times on 4/5 of
the records (out-of-fold predictions, so each student's personal correction is measured honestly), the shrinkage `k`
is chosen, the earned-trust gate is checked for every student, and the final forest is trained on all records.

In [ ]:
from components.c1_disruption.ml.drift.model import DriftModel
demo_model = DriftModel(n_estimators=100).fit(records, progress=progress)
progress(f"done: k = {demo_model.k}, gate open for {sum(t.trusted for t in demo_model.trust.values())} "
         f"of {len(demo_model.trust)} personas")

**Model vs baselines.** *Cold start* = leave one student out (the model has never seen them). *Warm* = rolling origin
(each week is predicted from earlier days only, so the model is retrained many times — the progress lines show each
student). B0 = trust the estimate, B1 = overall average error, B2 = each student's own average, B3 = average plus a
personal correction without the forest. MAE is in minutes (lower is better); coverage is the share of actual durations
inside the q10–q90 range (nominal 80%). *Gate open* = how often the earned-trust gate let the model correct; in cold
start it is always closed, so there the model equals B0.

In [ ]:
from components.c1_disruption.ml.drift import baselines
from components.c1_disruption.ml.drift.evaluation import leave_one_student_out, rolling_origin, score
methods = {"model": lambda: DriftModel(n_estimators=100), **{c.name: c for c in baselines.BASELINES}}
cold, warm = {}, {}
for name, factory in methods.items():
    progress(f"=== {name}: cold start + warm ===")
    steps = progress if name == "model" else None  # the baselines take no time: one line each
    cold[name] = leave_one_student_out(records, factory, progress=steps, label=f"{name}: ")
    warm[name] = rolling_origin(records, factory, 1, 7, progress=steps, label=f"{name}: ")
for name in methods:
    c, w = score(cold[name]), score(warm[name])
    gate = "" if w["trustedShare"] is None else f" | gate open {w['trustedShare']:.0%}"
    print(f"{name:22} cold MAE {c['maeMinutes']:6.2f} min | warm MAE {w['maeMinutes']:6.2f} min | "
          f"coverage {w['coverage'] if w['coverage'] is None else round(w['coverage'], 3)}{gate}")

In [ ]:
from components.c1_disruption.ml import charts
charts.mae_by_method({name: {"cold start": score(cold[name])["maeMinutes"], "warm": score(warm[name])["maeMinutes"]}
                      for name in methods}, "Model 1 vs baselines — MAE by method (SYNTHETIC)");

**Learning curve.** How the warm error changes with how many of their own tasks the student has finished.

In [ ]:
from components.c1_disruption.ml.drift.evaluation import learning_curve
charts.learning_curve({"Model 1 (warm)": learning_curve(warm["model"])}, "Model 1 learning curve (SYNTHETIC)");

**Predicted vs actual minutes** (warm predictions). Grey = the student's own estimate, blue = Model 1. Points on the
black line are perfect; Model 1 should sit closer to it than the grey points where a student has a real drift.

In [ ]:
import math
points = [(r["estimatedMinutes"], r["actualMinutes"], r["estimatedMinutes"] * math.exp(v))
          for r, v in zip(warm["model"].records, warm["model"].log_pred)]
charts.predicted_vs_actual(points, "Model 1: predicted vs actual minutes (SYNTHETIC, warm)");

**Earned-trust gate per student** (warm, at each student's last evaluation point): green = the model may correct this student.

In [ ]:
from components.c1_disruption.ml.drift.evaluation import gate_by_student
gate = gate_by_student(warm["model"])
charts.gate_per_student({s: float(v) for s, v in gate.items()}, "Earned-trust gate open per persona (SYNTHETIC, warm)");
print({s: "open" if v else "closed" for s, v in gate.items()})

**Save the trained model for the API (format 2).** This trains Model 1 once more on *all* the records (production
setting: 200 trees) and saves two files in the output folder:

- `c1_drift_model.joblib` — the trained model, **including the earned-trust gate** (for each student: whether the
  model beat their own estimate on their 20 most recent tasks) and the median point (q50);
- `c1_drift_model.joblib.json` — a small label: **format 2**, training date, number of records, `SYNTHETIC` flag, the
  shrinkage value k, the gate settings, how many students have the gate open, and library versions.

In Colab both files are downloaded to your computer. To use them in the C1 API, put them in one folder (for example
`components/c1_disruption/models/`) and set `C1_DRIFT_MODEL_PATH=components/c1_disruption/models/c1_drift_model.joblib`
in `.env`. Without that setting, or if the file is missing, the API plans with drift 1.0. A file made before
16 Oct 2026 (format 1) is refused (drift 1.0), so retrain here once. Model files are never committed.

In [ ]:
from components.c1_disruption.ml.drift.persist import FORMAT_VERSION, save_model
final_model = DriftModel().fit(records, progress=progress)
model_file = os.path.join(OUT, "c1_drift_model.joblib")
meta = save_model(final_model, model_file, dataset=world.dataset)
assert meta["format"] == FORMAT_VERSION == 2, meta["format"]
print(meta)
print("Earned-trust gate per student (model MAE vs own-estimate MAE on their recent tasks, minutes):")
for student, t in sorted(final_model.trust.items()):
    print(f"  {student}: {'OPEN  ' if t.trusted else 'closed'}  model {t.model_mae:.1f} vs B0 {t.b0_mae:.1f} ({t.n_recent} tasks)")
if IN_COLAB:
    from google.colab import files
    files.download(model_file)
    files.download(model_file + ".json")
else:
    print("Saved:", model_file, "and", model_file + ".json")